# Session 2 — Indexing, Selecting & Filtering

Runnable code for the **Build It** and **Experiment** sections.
Run each cell top to bottom.

In [1]:
import pandas as pd

print("pandas version:", pd.__version__)

pandas version: 3.0.3


In [2]:
sales = pd.DataFrame(
    {
        "city": ["Berlin", "Paris", "Berlin", "Lisbon", "Paris", "Berlin"],
        "price": [12.5, 30.0, 7.5, 22.0, 15.0, 40.0],
        "units": [120, 45, 300, 80, 200, 15],
    },
    index=["o1", "o2", "o3", "o4", "o5", "o6"],
)
sales

,city,price,units
o1,Berlin,12.5,120
o2,Paris,30.0,45
o3,Berlin,7.5,300
o4,Lisbon,22.0,80
o5,Paris,15.0,200
o6,Berlin,40.0,15


## 1. Build It — Core Code

### 1.1 `.loc` — select by label (row, then column)

In [3]:
print("loc cell      :", sales.loc["o3", "price"])
print("loc whole row :", sales.loc["o2"].to_dict())
print("loc whole col :", sales.loc[:, "price"].tolist())

loc cell      : 7.5
loc whole row : {'city': 'Paris', 'price': 30.0, 'units': 45}
loc whole col : [12.5, 30.0, 7.5, 22.0, 15.0, 40.0]


### 1.2 `.iloc` — select by integer position

In [4]:
print("iloc cell     :", sales.iloc[2, 1])
print("iloc first row:", sales.iloc[0].to_dict())

iloc cell     : 7.5
iloc first row: {'city': 'Berlin', 'price': 12.5, 'units': 120}


### 1.3 Slices: `.loc` is inclusive, `.iloc` is exclusive

In [5]:
print("loc  o2..o4 :", sales.loc["o2":"o4"].index.tolist())
print("iloc 0..3   :", sales.iloc[0:3].index.tolist())

loc  o2..o4 : ['o2', 'o3', 'o4']
iloc 0..3   : ['o1', 'o2', 'o3']


### 1.4 Boolean filtering

In [6]:
expensive = sales[sales["price"] > 20]
expensive

,city,price,units
o2,Paris,30.0,45
o4,Lisbon,22.0,80
o6,Berlin,40.0,15


### 1.5 Combining conditions with `&`, `|`, `~`

In [7]:
mask_and = sales[(sales["price"] > 15) & (sales["units"] < 100)]
print("price > 15 AND units < 100 ->", mask_and.index.tolist())

mask_or = sales[(sales["units"] > 100) | (sales["price"] < 10)]
print("units > 100 OR  price < 10 ->", mask_or.index.tolist())

mask_not = sales[~(sales["city"] == "Berlin")]
print("NOT Berlin                 ->", mask_not.index.tolist())

price > 15 AND units < 100 -> ['o2', 'o4', 'o6']
units > 100 OR  price < 10 -> ['o1', 'o3', 'o5']
NOT Berlin                 -> ['o2', 'o4', 'o5']


### 1.6 `isin`, `between`, `query`

In [8]:
print("isin   :", sales[sales["city"].isin(["Berlin", "Lisbon"])].index.tolist())
print("between:", sales[sales["price"].between(10, 25)].index.tolist())
print("query  :", sales.query("units > 100 or price < 10").index.tolist())

isin   : ['o1', 'o3', 'o4', 'o6']
between: ['o1', 'o4', 'o5']
query  : ['o1', 'o3', 'o5']


### 1.7 Assigning with `.loc` (Copy-on-Write-safe)

In [9]:
sales.loc[sales["units"] > 100, "status"] = "high"
sales.loc[sales["units"] <= 100, "status"] = "normal"
sales

,city,price,units,status
o1,Berlin,12.5,120,high
o2,Paris,30.0,45,normal
o3,Berlin,7.5,300,high
o4,Lisbon,22.0,80,normal
o5,Paris,15.0,200,high
o6,Berlin,40.0,15,normal


### 1.8 Sorting with `sort_values`

In [10]:
sales.sort_values(["city", "units"], ascending=[True, False])

,city,price,units,status
o3,Berlin,7.5,300,high
o1,Berlin,12.5,120,high
o6,Berlin,40.0,15,normal
o4,Lisbon,22.0,80,normal
o5,Paris,15.0,200,high
o2,Paris,30.0,45,normal


## 2. Experiment

Change a parameter, predict the output, then run the cell.

**Experiment 1** — `.loc` inclusive vs `.iloc` exclusive on a single column.

In [11]:
print("loc  o1..o3 price:", sales.loc["o1":"o3", "price"].tolist())
print("iloc 0..3 price  :", sales.iloc[0:3, 1].tolist())

loc  o1..o3 price: [12.5, 30.0, 7.5]
iloc 0..3 price  : [12.5, 30.0, 7.5]


**Experiment 2** — `and` raises, and `&` without parentheses silently misbehaves.
Because `&` binds tighter than `>`, the expression parses as
`price > (15 & mask)`, which keeps *every* row instead of filtering.

In [12]:
try:
    sales[(sales["price"] > 15) and (sales["units"] < 100)]
except ValueError as err:
    print("ValueError:", err)

without_parens = sales[sales["price"] > 15 & (sales["units"] < 100)]
with_parens = sales[(sales["price"] > 15) & (sales["units"] < 100)]
print("without parentheses:", without_parens.index.tolist())
print("with    parentheses:", with_parens.index.tolist())

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
without parentheses: ['o1', 'o2', 'o3', 'o4', 'o5', 'o6']
with    parentheses: ['o2', 'o4', 'o6']


**Experiment 3** — `query` understands `and`, `or`, and `not`.

In [13]:
sales.query("price >= 10 and price <= 30 and city != 'Lisbon'")

,city,price,units,status
o1,Berlin,12.5,120,high
o2,Paris,30.0,45,normal
o5,Paris,15.0,200,high


**Experiment 4** — `between` is inclusive on both ends.

In [14]:
print("between(12.5, 22.0):", sales[sales["price"].between(12.5, 22.0)].index.tolist())

between(12.5, 22.0): ['o1', 'o4', 'o5']


**Experiment 5** — `sort_values` returns a new frame; the original is untouched.

In [15]:
sorted_view = sales.sort_values("price", ascending=False)
print("sorted first row:", sorted_view.index[0])
print("original first  :", sales.index[0])
print("original dtype  :", sales["price"].dtype)

sorted first row: o6
original first  : o1
original dtype  : float64


## 3. Mini Project — Sales Explorer

In [16]:
orders = pd.DataFrame(
    {
        "order_id": ["A100", "A101", "A102", "A103", "A104", "A105", "A106", "A107"],
        "city": ["Berlin", "Paris", "Berlin", "Lisbon", "Paris", "Berlin", "Lisbon", "Paris"],
        "product": ["Widget", "Gadget", "Gizmo", "Widget", "Gadget", "Gizmo", "Gadget", "Widget"],
        "price": [12.5, 30.0, 7.5, 22.0, 15.0, 40.0, 18.0, 26.0],
        "units": [120, 45, 300, 80, 200, 15, 60, 25],
    }
)
orders

,order_id,city,product,price,units
0,A100,Berlin,Widget,12.5,120
1,A101,Paris,Gadget,30.0,45
2,A102,Berlin,Gizmo,7.5,300
3,A103,Lisbon,Widget,22.0,80
4,A104,Paris,Gadget,15.0,200
5,A105,Berlin,Gizmo,40.0,15
6,A106,Lisbon,Gadget,18.0,60
7,A107,Paris,Widget,26.0,25


In [17]:
# Step 2 — a specific order by label.
orders.loc[orders["order_id"] == "A100"]

,order_id,city,product,price,units
0,A100,Berlin,Widget,12.5,120


In [18]:
# Step 3 — positional slices, two columns only.
print("last 3 rows:")
print(orders.iloc[-3:, [1, 3]])
print("\nfirst 4 rows:")
print(orders.iloc[:4, [1, 3]])

last 3 rows:
     city  price
5  Berlin   40.0
6  Lisbon   18.0
7   Paris   26.0

first 4 rows:
     city  price
0  Berlin   12.5
1   Paris   30.0
2  Berlin    7.5
3  Lisbon   22.0


In [19]:
# Step 4 — combine two conditions.
big_orders = orders[(orders["price"] > 20) & (orders["units"] > 10)]
big_orders

,order_id,city,product,price,units
1,A101,Paris,Gadget,30.0,45
3,A103,Lisbon,Widget,22.0,80
5,A105,Berlin,Gizmo,40.0,15
7,A107,Paris,Widget,26.0,25


In [20]:
# Step 5 — isin and between.
print(orders[orders["city"].isin(["Berlin", "Lisbon"])][["order_id", "city"]])
print()
print(orders[orders["price"].between(10, 30)][["order_id", "price"]])

  order_id    city
0     A100  Berlin
2     A102  Berlin
3     A103  Lisbon
5     A105  Berlin
6     A106  Lisbon

  order_id  price
0     A100   12.5
1     A101   30.0
3     A103   22.0
4     A104   15.0
6     A106   18.0
7     A107   26.0


In [21]:
# Step 6 — conditional assignment with .loc.
orders.loc[orders["price"] >= 25, "tier"] = "premium"
orders.loc[orders["price"] < 25, "tier"] = "standard"

# Step 7 — derive revenue and sort.
orders["revenue"] = orders["price"] * orders["units"]
orders.sort_values("revenue", ascending=False).head(3)

,order_id,city,product,price,units,tier,revenue
4,A104,Paris,Gadget,15.0,200,standard,3000.0
2,A102,Berlin,Gizmo,7.5,300,standard,2250.0
3,A103,Lisbon,Widget,22.0,80,standard,1760.0


In [22]:
# Step 8 — revenue per city (teaser for Session 4).
orders.groupby("city")["revenue"].sum()

city
Berlin    4350.0
Lisbon    2840.0
Paris     5000.0
Name: revenue, dtype: float64